This version of SLIMER implementation eliminates the line
`from src.SFT_finetuning.commons.prompter import SLIMER_instruction_prompter, Prompter`
and instead builds a function unreliant on cloning SLIMER

In [2]:
!git clone https://github.com/jyjylow/mapping_reform.git

Cloning into 'mapping_reform'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 94 (delta 28), reused 69 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 27.99 MiB | 12.25 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [3]:
!pip3 install vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.4/383.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.0/169.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7

In [18]:
import os
import sys
sys.path.append(os.path.abspath("./SLIMER"))
import json

from vllm import LLM, SamplingParams
# from src.SFT_finetuning.commons.prompter import SLIMER_instruction_prompter, Prompter

# to truncate decoder prompt at 512 tokens, using SLIMER's tokenizer
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("expertai/SLIMER")

In [5]:
# loading SLIMER model

vllm_model = LLM("expertai/SLIMER", max_model_len=512)
# it is recommended to use a temperature of 0
# max_new_tokens can be adjusted depending on the expected length and number of entities (default 128)
sampling_params = SamplingParams(temperature=0, max_tokens=128, stop=['</s>'])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/734 [00:00<?, ?B/s]

INFO 07-18 17:54:51 [config.py:841] This model supports multiple tasks: {'embed', 'reward', 'generate', 'classify'}. Defaulting to 'generate'.
WARNING 07-18 17:54:51 [config.py:3320] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 07-18 17:54:51 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 07-18 17:54:51 [config.py:1472] Using max model len 512
WARNING 07-18 17:54:51 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 07-18 17:54:51 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='expertai/SLIMER', speculative_config=None, tokenizer='expertai/SLIMER', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 07-18 17:59:16 [default_loader.py:272] Loading weights took 45.88 seconds
INFO 07-18 17:59:17 [model_runner.py:1203] Model loading took 12.5524 GiB and 261.791814 seconds
INFO 07-18 17:59:20 [worker.py:294] Memory profiling takes 2.25 seconds
INFO 07-18 17:59:20 [worker.py:294] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.90) = 13.27GiB
INFO 07-18 17:59:20 [worker.py:294] model weights take 12.55GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 0.31GiB; the rest of the memory reserved for KV Cache is 0.37GiB.
INFO 07-18 17:59:21 [executor_base.py:113] # cuda blocks: 47, # CPU blocks: 512
INFO 07-18 17:59:21 [executor_base.py:118] Maximum concurrency for 512 tokens per request: 1.47x
INFO 07-18 17:59:23 [model_runner.py:1513] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in th

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 07-18 18:00:03 [model_runner.py:1671] Graph capturing finished in 40 secs, took 0.26 GiB
INFO 07-18 18:00:03 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 45.74 seconds


In [27]:

entity_definitions = {
    "event": {
        "definition": "EVENT entities refer to specific manifestations of collective action for sociopolitical movements. These can be nouns such as meetings, protests, demonstrations, lectures, and gatherings, or verbs describing these actions like met, protested, demonstrated, lectured, and gathered.",
        "guidelines": "When a verb like 'met' is present, extract the nominal form 'meeting'. Avoid labeling speculative, fictional, or allegorical events. Label only events that have occurred and are being factually recounted. "
    },
    "location": {
        "definition": "LOCATION entities refer to geographic places and proper nouns of specific locations, such as cities, counties, districts, and named buildings or places.",
        "guidelines": "Focus on geographic and administrative divisions, proper place names, and specific addresses. Extract as much locational information as is available. Example: extract all of 'Ship Inn, Long Lane, Bermondsey' rather than just 'Bermondsey'. Avoid labeling building types or generic spatial categories."
    },
    "space": {
        "definition": "SPACE entities describe types of physical spaces, buildings, or venues where people gather or activities take place.",
        "guidelines": "Look for words like: meeting room, school, factory, hall, building, house, shop, office, church, town, square, etc. Extract these words even when they appear in proper names like 'People's School' (extract 'school') or 'Market Square' (extract 'square'). Focus on the common noun that describes the type of space."
        }
}

# The new, self-contained function
def extract_entities(input_text, max_length=512):
    """
    Extracts entities using manually formatted prompts without relying on
    the SLIMER repository's helper scripts.
    """
    print(f"INPUT TEXT: \"{input_text}\"")
    print("\nENTITIES:")

    all_outputs = {}

    for tag, dng in entity_definitions.items():

        # 1. Creating the instruction string
        instruction = (
            f"Extract the Named Entities of type {tag.upper()} from the input text. "
            f"You are given a DEFINITION and some GUIDELINES.\n"
            f"DEFINITION: {dng['definition']}\n"
            f"GUIDELINES: {dng['guidelines']}\n"
            "Return a JSON list of instances of this Named Entity type. "
            "Return an empty list if no instances are present."
        )

        # 2. Constructing the prompt scaffold
        prompt_scaffold = (
            "[INST] You are given a text chunk (delimited by triple quotes) and an instruction.\n"
            "Read the text and answer to the instruction in the end.\n"
            '"""\n'
            f"{input_text}\n"
            '"""\n'
            f"Instruction: {instruction}\n"
            "[/INST]"
        )

        # 3. Calculating token budget for input_text
        scaffold_without_inputtext = prompt_scaffold.replace("{input_text}\n", "")
        scaffold_token_count = len(tokenizer.encode(scaffold_without_inputtext))
        print(f"No. of tokens in prompt scaffold: {scaffold_token_count}")
        budget_for_input_text = max_length - scaffold_token_count - 5 # Leaving a small buffer for safety

        # 4. Tokenizing and truncating the input_text if necessary
        input_tokens = tokenizer.encode(input_text)
        if len(input_tokens) > budget_for_input_text:
            print(f"Input text is too long ({len(input_tokens)} tokens), truncating to {budget_for_input_text} tokens.")
            truncated_input_tokens = input_tokens[:budget_for_input_text]
            truncated_input_text = tokenizer.decode(truncated_input_tokens, skip_special_tokens=True)
        else:
            truncated_input_text = input_text

        # 5. Assembling the final prompt with the (potentially truncated) text
        final_prompt = prompt_scaffold.replace("{input_text_placeholder}", truncated_input_text)

        # 6. Generating responses
        responses = vllm_model.generate([final_prompt], sampling_params)
        pred_response = responses[0].outputs[0].text.strip()

        # Parse the JSON output
        try:
            entities = json.loads(pred_response) if pred_response else []
        except json.JSONDecodeError:
            entities = []  # Handle cases where the model returns non-JSON text

        all_outputs[tag] = entities

        # Format output
        if entities:
            entities_str = ", ".join([f'"{entity}"' for entity in entities])
            print(f"{tag.upper()}: [{entities_str}]")
        else:
            print(f"{tag.upper()}: []")

    return all_outputs

In [30]:
extract_entities(input_text)

INPUT TEXT: "STALYBRIDGE.—A public meeting was held in the People’s School here on Monday evening last, when the National Petition was read and adopted; after which, Mr. James Leach, of Manchester, delivered an address, exposing the fallacies of the Corn Law repealers. A Corn Law lecture had been previously delivered in the town, by a Mr. Spencer, to about half a dozen of the middle classes; the Chartists, however, upset his meeting."

ENTITIES:
No. of tokens in prompt scaffold: 339


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

EVENT: []
No. of tokens in prompt scaffold: 323


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

LOCATION: []
No. of tokens in prompt scaffold: 324


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

SPACE: []


{'event': [], 'location': [], 'space': []}

In [31]:
simple_test = """
Weavers and card-room hands, attend the meeting which will be held in the Charlestown meeting room, on Wednesday evening, Nov. 6th, at eight o'clock, and show by your thousands that you are determined to be no longer 'stumped upon with impunity.
"""

extract_entities(simple_test)

INPUT TEXT: "
Weavers and card-room hands, attend the meeting which will be held in the Charlestown meeting room, on Wednesday evening, Nov. 6th, at eight o'clock, and show by your thousands that you are determined to be no longer 'stumped upon with impunity.
"

ENTITIES:
No. of tokens in prompt scaffold: 302


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

EVENT: ["meeting"]
No. of tokens in prompt scaffold: 286


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

LOCATION: ["Charlestown"]
No. of tokens in prompt scaffold: 287


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

SPACE: ["Charlestown meeting room"]


{'event': ['meeting'],
 'location': ['Charlestown'],
 'space': ['Charlestown meeting room']}

In [29]:
test_text = """
A comfortable supper party met at the Chequers Inn in the evening, but Mr. O'Connor could not be present. '
"""
extract_entities(test_text)

INPUT TEXT: "
A comfortable supper party met at the Chequers Inn in the evening, but Mr. O'Connor could not be present. '
"

ENTITIES:
No. of tokens in prompt scaffold: 266


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

EVENT: ["supper party"]
No. of tokens in prompt scaffold: 250


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

LOCATION: ["Chequers Inn"]
No. of tokens in prompt scaffold: 251


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

SPACE: []


{'event': ['supper party'], 'location': ['Chequers Inn'], 'space': []}

In [28]:
test_short = """
OLD ABERDEEN.—On Wednesday night week, a public meeting was held in the Teetotal Hall, High-street, for the purpose of forming a Chartist Association, at eight o'clock. The Hall was crowded. Mr. William Adams was called to the chair, who opened the business in an appropriate and pithy address, and introduced Mr. Nicolson, from Aberdeen. Mr. Nicolson delivered an address on the present state of the country, &c., and sat down warmly applauded. Mr. Archibald Macdonald then explained the principles of the Charter, and was followed by Mr. James Macpherson, who delivered a powerful address on the necessity of uniting in one common bond of union to overturn the unjust system of things which now exists. A gentleman named Mr. Gibb then put some questions to the speakers, which were answered to his seeming satisfaction. The National Petition, and copies of the Charter, were distributed, and an Association formed. A vote of thanks was given to the Chairman, and the meeting separated.  CHESTER.—Mr. Christopher Doyle lectured here on Thursday night week, at seven o'clock, in the Chartist Meeting Room, Steam Mill-street. Admission gratis, and free discussion was invited. The room, which will hold between 300 and 400 persons, was crowded. Thanks were voted to him at the close, and eight new members were enrolled. The National Petition was adopted at a public meeting on Monday night last."""
extract_entities(test_short)

INPUT TEXT: "
OLD ABERDEEN.—On Wednesday night week, a public meeting was held in the Teetotal Hall, High-street, for the purpose of forming a Chartist Association, at eight o'clock. The Hall was crowded. Mr. William Adams was called to the chair, who opened the business in an appropriate and pithy address, and introduced Mr. Nicolson, from Aberdeen. Mr. Nicolson delivered an address on the present state of the country, &c., and sat down warmly applauded. Mr. Archibald Macdonald then explained the principles of the Charter, and was followed by Mr. James Macpherson, who delivered a powerful address on the necessity of uniting in one common bond of union to overturn the unjust system of things which now exists. A gentleman named Mr. Gibb then put some questions to the speakers, which were answered to his seeming satisfaction. The National Petition, and copies of the Charter, were distributed, and an Association formed. A vote of thanks was given to the Chairman, and the meeting separated

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: The decoder prompt (length 576) is longer than the maximum model length of 512. Make sure that `max_model_len` is no smaller than the number of text tokens.

In [54]:
# EXTRACT ENTITIES FUNCTION V2: ALL IN ONE BABY

def extract_entities_all(input_text, max_length=512):
    """
    Extracts all defined entities in a single, efficient model call
    using a comprehensive all-in-one prompt. This version has corrected indentation.
    """

    # 1. Combining all entities into one instruction
    instruction = (
        "From the text chunk, extract Named Entities for the following types:\n"
        "- EVENT: A collective action for a sociopolitical movement (e.g., meeting, protest, lecture). Extract verbs like 'met' as their noun form.\n"
        "- LOCATION: A specific geographic place (e.g., city, street, named building). Extract full proper place names.\n"
        "- SPACE: A generic type of place where people gather (e.g., room, hall, school). Extract the common noun describing the space.\n\n"
        'Return a single JSON object with keys "event", "location", and "space". Each key should have a list of strings as its value. '
        "If no instances of an entity type are found, return an empty list for that key."
    )

    # 2. Constructing prompt scaffold
    prompt_scaffold = (
        "[INST] You are given a text chunk (delimited by triple quotes) and an instruction.\n"
        "Read the text and answer to the instruction in the end.\n"
        '"""\n{input_text_placeholder}\n"""\n'
        f"Instruction: {instruction}\n[/INST]"
    )

    # 3. Calculating the token budget for the input_text
    scaffold_without_placeholder = prompt_scaffold.replace("{input_text_placeholder}\n", "")
    scaffold_token_count = len(tokenizer.encode(scaffold_without_placeholder))
    print(f"INFO: All-in-one scaffold token count: {scaffold_token_count}")
    budget_for_input_text = max_length - scaffold_token_count - 5 # Safety buffer

    # 4. Truncate the input text if it exceeds the budget
    input_tokens = tokenizer.encode(input_text)
    if len(input_tokens) > budget_for_input_text:
        print(f"INFO: Input text is too long ({len(input_tokens)} tokens), truncating to {budget_for_input_text} tokens.")
        truncated_input_tokens = input_tokens[:budget_for_input_text]
        truncated_input_text = tokenizer.decode(truncated_input_tokens, skip_special_tokens=True)
    else:
        truncated_input_text = input_text

    # 5. Assemble the final prompt
    final_prompt = prompt_scaffold.replace("{input_text_placeholder}", truncated_input_text)

    # 6. Generate a SINGLE response from the model
    responses = vllm_model.generate([final_prompt], sampling_params)
    pred_response = responses[0].outputs[0].text.strip()

    # 7. Parse the single JSON output robustly
    try:
        all_outputs = json.loads(pred_response)
        # Ensure the structure is what we expect, adding missing keys if necessary
        for key in ["event", "location", "space"]:
            if key not in all_outputs:
                all_outputs[key] = []
    except (json.JSONDecodeError, TypeError):
        print("Error: Model did not return a valid JSON object.")
        all_outputs = {"event": [], "location": [], "space": []} # Default empty structure

    print("\nENTITIES:")
    for tag, entities in all_outputs.items():
        if entities:
          entities_str = ", ".join([f'"{entity}"' for entity in entities])
          print(f"{tag.upper()}: [{entities_str}]")

        else:
            print(f"{tag.upper()}: []")

    return all_outputs

In [55]:
extract_entities_all(test_text)

INFO: All-in-one scaffold token count: 220


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Error: Model did not return a valid JSON object.

ENTITIES:
EVENT: []
LOCATION: []
SPACE: []


{'event': [], 'location': [], 'space': []}

Looks like all-in-one isn't gonna work. I'll stick to the OG for now.